In [1]:
from gridstatusio import GridStatusClient
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tomli

ModuleNotFoundError: No module named 'gridstatusio'

In [ ]:
with open("config.toml", "rb") as f:
    config = tomli.load(f)

# 2. Extract the key and initialize the client
api_key = config["api_keys"]["gridstatus"]
client = GridStatusClient(api_key=api_key)

# 3. Verify it works
print(client.get_api_usage())

{'plan_name': 'Free', 'limits': {'api_rows_returned_limit': 500000, 'api_requests_limit': 250, 'api_rows_per_response_limit': 50000, 'per_second_api_rate_limit': 1, 'per_minute_api_rate_limit': 30, 'per_hour_api_rate_limit': 600}, 'current_usage_period_start': '2026-10-01T00:00:00Z', 'current_usage_period_end': '2026-11-01T00:00:00Z', 'current_period_usage': {'total_requests': 16, 'total_api_rows_returned': 20640}}


In [ ]:
meta = client.get_dataset_metadata("ercot_mcpc_dam")

print("Available from:", meta["earliest_available_time_utc"])
print("Available columns:")
print([c["name"] for c in meta["all_columns"]])

Available from: 2010-11-29 06:00:00+00:00
Available columns:
['interval_start_utc', 'interval_end_utc', 'as_type', 'mcpc']


In [ ]:
test = client.get_dataset(
    "ercot_mcpc_dam",
    start="2025-11-20",
    end="2025-11-21",
    timezone="market",
    limit=250
)

print(test.shape)
print(test.columns.tolist())
print(test.head())

2026-10-21 13:19:18 - INFO - Fetching Page 1...
2026-10-21 13:19:18 - INFO - GET https://api.gridstatus.io/v1/datasets/ercot_mcpc_dam/query
2026-10-21 13:19:18 - INFO - Params: {'start_time': Timestamp('2025-11-20 00:00:00'), 'end_time': Timestamp('2025-11-21 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': 250, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': 'market', 'cursor': '', 'filter_column': None, 'filter_value': None, 'filter_operator': '=', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-10-21 13:19:18 - INFO - Done in 0.3 seconds. 
2026-10-21 13:19:18 - INFO - Total rows: 120/250 (48.0% of limit)
2026-10-21 13:19:18 - INFO - Total number of rows: 120


(120, 6)
['interval_start_local', 'interval_start_utc', 'interval_end_local', 'interval_end_utc', 'as_type', 'mcpc']
       interval_start_local        interval_start_utc        interval_end_local          interval_end_utc  \
0 2025-11-20 00:00:00-06:00 2025-11-20 06:00:00+00:00 2025-11-20 01:00:00-06:00 2025-11-20 07:00:00+00:00   
1 2025-11-20 00:00:00-06:00 2025-11-20 06:00:00+00:00 2025-11-20 01:00:00-06:00 2025-11-20 07:00:00+00:00   
2 2025-11-20 00:00:00-06:00 2025-11-20 06:00:00+00:00 2025-11-20 01:00:00-06:00 2025-11-20 07:00:00+00:00   
3 2025-11-20 00:00:00-06:00 2025-11-20 06:00:00+00:00 2025-11-20 01:00:00-06:00 2025-11-20 07:00:00+00:00   
4 2025-11-20 00:00:00-06:00 2025-11-20 06:00:00+00:00 2025-11-20 01:00:00-06:00 2025-11-20 07:00:00+00:00   

  as_type  mcpc  
0    ECRS  0.37  
1   NSPIN  0.49  
2   REGDN  0.47  
3   REGUP  0.75  
4     RRS  0.50  


In [ ]:
# The end date is exclusive.
windows = [
    ("2024", "Earlier", "2024-11-20", "2024-12-04"),
    ("2024", "Later",   "2024-12-06", "2024-12-20"),
    ("2025", "Earlier", "2025-11-20", "2025-12-04"),
    ("2025", "Later",   "2025-12-06", "2025-12-20"),
]

frames = []

for year, window, start, end in windows:
    df = client.get_dataset(
        "ercot_mcpc_dam",
        start=start,
        end=end,
        timezone="market",
        limit=2000
    )

    expected_rows = 14 * 24 * 5 # 14 days, 24 hours, 5 products


    if len(df) != expected_rows:
        print(f"Warning: Expected {expected_rows} rows, got {len(df)}")

    df["year"] = year
    df["window"] = window
    frames.append(df)

    print(year, window, len(df), "rows")

prices = pd.concat(frames, ignore_index=True)

# Verify observations per product and window
print(
    prices.groupby(["year", "window", "as_type"])
    .size()
)

prices.to_csv("ercot_mcpc_comparison.csv", index=False)


2026-10-21 21:18:30 - INFO - Fetching Page 1...
2026-10-21 21:18:30 - INFO - GET https://api.gridstatus.io/v1/datasets/ercot_mcpc_dam/query
2026-10-21 21:18:30 - INFO - Params: {'start_time': Timestamp('2024-11-20 00:00:00'), 'end_time': Timestamp('2024-12-04 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': 2000, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': 'market', 'cursor': '', 'filter_column': None, 'filter_value': None, 'filter_operator': '=', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-10-21 21:18:30 - INFO - Done in 0.23 seconds. 
2026-10-21 21:18:30 - INFO - Total rows: 1,680/2,000 (84.0% of limit)
2026-10-21 21:18:30 - INFO - Total number of rows: 1680
2026-10-21 21:18:30 - INFO - Fetching Page 1...
2026-10-21 21:18:30 - INFO - GET https://api.gridstatus.io/v1/datasets/ercot_mcpc_dam/query
2026-10-21 21:18:30 - INFO - Params: {'start_time

2024 Earlier 1680 rows


2026-10-21 21:18:32 - INFO - Done in 2.33 seconds. 
2026-10-21 21:18:32 - INFO - Total rows: 1,680/2,000 (84.0% of limit)
2026-10-21 21:18:32 - INFO - Total number of rows: 1680
2026-10-21 21:18:32 - INFO - Fetching Page 1...
2026-10-21 21:18:32 - INFO - GET https://api.gridstatus.io/v1/datasets/ercot_mcpc_dam/query
2026-10-21 21:18:32 - INFO - Params: {'start_time': Timestamp('2025-11-20 00:00:00'), 'end_time': Timestamp('2025-12-04 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': 2000, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': 'market', 'cursor': '', 'filter_column': None, 'filter_value': None, 'filter_operator': '=', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-10-21 21:18:33 - INFO - Too Many Requests. Limit: 1 per 1 second.. Retrying in 2 seconds. Retry 1 of 5.


2024 Later 1680 rows


2026-10-21 21:18:35 - INFO - Done in 2.37 seconds. 
2026-10-21 21:18:35 - INFO - Total rows: 1,680/2,000 (84.0% of limit)
2026-10-21 21:18:35 - INFO - Total number of rows: 1680
2026-10-21 21:18:35 - INFO - Fetching Page 1...
2026-10-21 21:18:35 - INFO - GET https://api.gridstatus.io/v1/datasets/ercot_mcpc_dam/query
2026-10-21 21:18:35 - INFO - Params: {'start_time': Timestamp('2025-12-06 00:00:00'), 'end_time': Timestamp('2025-12-20 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': 2000, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': 'market', 'cursor': '', 'filter_column': None, 'filter_value': None, 'filter_operator': '=', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-10-21 21:18:35 - INFO - Too Many Requests. Limit: 1 per 1 second.. Retrying in 2 seconds. Retry 1 of 5.


2025 Earlier 1680 rows


2026-10-21 21:18:37 - INFO - Done in 2.37 seconds. 
2026-10-21 21:18:37 - INFO - Total rows: 1,680/2,000 (84.0% of limit)
2026-10-21 21:18:37 - INFO - Total number of rows: 1680


2025 Later 1680 rows
year  window   as_type
2024  Earlier  ECRS       336
               NSPIN      336
               REGDN      336
               REGUP      336
               RRS        336
      Later    ECRS       336
               NSPIN      336
               REGDN      336
               REGUP      336
               RRS        336
2025  Earlier  ECRS       336
               NSPIN      336
               REGDN      336
               REGUP      336
               RRS        336
      Later    ECRS       336
               NSPIN      336
               REGDN      336
               REGUP      336
               RRS        336
dtype: int64


In [3]:

sns.set_theme(style="whitegrid")

g = sns.catplot(
    data=prices,
    kind="box",
    x="year",
    y="mcpc",
    hue="window",
    hue_order=["Earlier", "Later"],
    col="as_type",
    col_wrap=3,
    sharey=False,
    showfliers=False,
    height=3.5,
    aspect=1.1
)

g.set_titles("{col_name}")
g.set_axis_labels(
    "Year",
    "Day-ahead clearing price ($/MW-hour)"
)

g.fig.suptitle(
    "ERCOT Ancillary Service Prices: "
    "Seasonal Change in 2024 vs. 2025",
    fontsize=16
)

g.fig.subplots_adjust(top=0.89, bottom=0.13)

g.fig.text(
    0.5, 0.035,
    "Earlier = Nov 20–Dec 3    |    "
    "Later = Dec 6–19\n"
    "RTC+B went live December 5, 2025",
    ha="center",
    fontsize=11
)

plt.show()


NameError: name 'sns' is not defined

In [ ]:
summary = (
    prices.groupby(["as_type", "period"])["mcpc"]
    .agg(
        mean="mean",
        median="median",
        p95=lambda x: x.quantile(0.95)
    )
    .round(2)
)

print(summary)

KeyError: 'period'

In [ ]:
# The end date is exclusive.
windows = [
("2024", "Earlier", "2024-11-20", "2024-12-04"),
("2024", "Later", "2024-12-06", "2024-12-20"),
("2025", "Earlier", "2025-11-20", "2025-12-04"),
("2025", "Later", "2025-12-06", "2025-12-20"),
]


frames = []


for year, window, start, end in windows:
    df = client.get_dataset(
        "ercot_mcpc_dam",
        start=start,
        end=end,
        timezone="market",
        limit=2000
)


expected_rows = 14 * 24 * 5 # 14 days, 24 hours, 5 products



if len(df) != expected_rows:
    print(f"Warning: Expected {expected_rows} rows, got {len(df)}")


df["year"] = year
df["window"] = window
frames.append(df)


print(year, window, len(df), "rows")


prices = pd.concat(frames, ignore_index=True)


# Verify observations per product and window
print(
prices.groupby(["year", "window", "as_type"])
.size()
)


prices.to_csv("ercot_mcpc_comparison.csv", index=False)